### 实验前的环境准备

这一格不是正式建模代码，而是实验开始前的“环境体检”。

为什么要先运行它？

1. 同一份 notebook 换一台电脑、换一个 Python 环境后，常见问题不是算法写错，而是缺少库。
2. 如果一开始不先检查，后面往往会在 `import`、画图、读文件或训练模型时突然报错，初学者很难判断问题到底出在代码还是环境。
3. 现在这一格已经升级为“先检查、再自动安装”，目的就是把环境问题尽量提前解决。

运行后你会看到几类信息：

1. 当前 notebook 实际使用的是哪个 Python 解释器。
2. 已经检测到哪些核心库。
3. 如果有缺失库，系统会尝试自动安装。
4. 如果安装完成后仍未生效，通常只需要重启内核，再从第 1 格重新运行。

可以把这一格理解成：正式做实验前，先把工具箱点一遍，缺什么先补什么。这样后面的每一步更容易顺利完成，也更符合真实开发中的工作流程。

In [ ]:
import importlib
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'plotly': 'plotly',
    'nbformat': 'nbformat',
    'tqdm': 'tqdm',
    'IPython': 'ipython',
    'imblearn': 'imbalanced-learn',
}

note_lines = [
    '说明：本教程如果要使用 SMOTE 做类别平衡，需要 imbalanced-learn。',
    '说明：本教程使用 Plotly 在 notebook 中显示三维图，因此建议同时具备 nbformat。',
]


def is_module_available(module_name):
    return importlib.util.find_spec(module_name) is not None


installed_modules = []
missing_packages = []
for module_name, package_name in required_packages.items():
    if is_module_available(module_name):
        installed_modules.append(module_name)
    else:
        missing_packages.append(package_name)

print('当前 Python 解释器:', sys.executable)
print('已检测到的模块:', ', '.join(installed_modules) if installed_modules else '无')

if missing_packages:
    print('\n检测到缺失依赖，开始自动安装:')
    print('pip install ' + ' '.join(missing_packages))
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
        importlib.invalidate_caches()
        still_missing = [
            package_name
            for module_name, package_name in required_packages.items()
            if not is_module_available(module_name)
        ]
        if still_missing:
            print('\n以下依赖安装后仍未检测到，请重启内核后重试:')
            print(', '.join(still_missing))
        else:
            print('\n依赖已自动安装完成，可以继续运行本教程。')
    except subprocess.CalledProcessError as error:
        print(f'\n自动安装失败，返回码: {error.returncode}')
        print('请手动执行:')
        print('pip install ' + ' '.join(missing_packages))
else:
    print('\n依赖检查通过，可以继续运行本教程。')

for note in note_lines:
    print(note)

print('如果你已经安装过，但这里仍显示缺失，通常是因为当前 notebook 内核和安装库使用的 Python 环境不是同一个。')
print('如果刚完成自动安装，后续单元格仍报导入错误，重启内核后再从头运行一次。')

### 常规金融欺诈交易判别实践

本实验保留原来的金融欺诈检测主题和本地 CSV 数据读取方式，同时把讲解结构调整为与医学分类教程一致的节奏：先读取本地文件，再检查原始表、整理教学子集、构造风险特征，最后再进行 PyTorch 分类建模与结果分析。

- 数据集：PaySim 风格金融交易欺诈数据教学子集
- 本地文件：financial_data.csv
- 任务：根据交易类型、金额、账户余额变化等信息判断一笔交易是否存在欺诈风险
- 数据集简介：该数据集保留了真实风控任务中常见的类别不平衡、金额跨度大、交易前后余额变化复杂等特点，适合讲解分类建模与业务代价分析
- 优点：场景鲜明、风险信号丰富、与医学分类教程共享相同建模框架，便于横向比较不同分类任务

#### 本实验建议关注 4 个问题

1. 为什么原始金融数据不能直接全部送进模型，而要先整理教学子集？
2. 为什么要对交易类型做编码，还要额外构造余额变化、比例和误差类特征？
3. 为什么类别不平衡会让表面准确率看起来不错，但真实识别能力未必够用？
4. 为什么阈值变化会同时影响误判和漏判，风控场景应该更关注哪类错误？

### 开始前先建立分类概念框架

为了让常规版和医疗版的分类教程能直接对照，先把这两个实验共享的几个核心概念说清楚：

1. 分类任务预测的不是连续数值，而是“属于某一类的概率”，本章里分别是糖尿病高风险和欺诈交易风险。
2. `Sigmoid()` 会把模型输出压到 0 到 1 之间，所以我们可以把结果解释成“像正类的程度”。
3. 阈值决定“多大概率才判成正类”，阈值变低通常召回率更高，但误报也更容易增加。
4. 分类模型常用 `BCELoss()` 这类损失函数来衡量概率预测与真实标签的差距，模型会据此做反向更新。
5. `accuracy` 不是唯一指标，还要结合混淆矩阵、召回率和误判样本一起看；`learning_rate`、`epochs`、`threshold` 都会改变最终表现。

先有这套概念框架，后面再看代码，学生就更容易理解“为什么同一套 MLP，在不同场景里关注的评价重点不同”。

### 第一步：准备实验工具

这一格相当于把欺诈检测实验要用的“工具箱”先打开。

后面会用到的工具主要有：

1. `pandas`、`numpy`：处理交易记录表。
2. `torch`：搭建并训练分类模型。
3. `sklearn`：做标准化、数据切分和分类评价。
4. `imblearn`：处理类别不平衡问题。
5. `matplotlib`、`plotly`：帮助我们观察分类结果和误判分布。

对初学者来说，先知道这些工具各司其职，比一下子记住全部语法更重要。

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly.express as px
import plotly.io as pio
from tqdm import trange

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False

def configure_matplotlib_for_cjk():
    preferred_fonts = [
        "PingFang SC",
        "Hiragino Sans GB",
        "Heiti SC",
        "STHeiti",
        "Songti SC",
        "Arial Unicode MS",
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "WenQuanYi Zen Hei",
        "DejaVu Sans",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = font_name
            break
    plt.rcParams["axes.unicode_minus"] = False

configure_matplotlib_for_cjk()
pio.renderers.default = "plotly_mimetype"

In [ ]:
from pathlib import Path
from IPython.display import Image, display

image_path = Path('1.png')
if image_path.exists():
    display(Image(filename=str(image_path)))
else:
    print('未找到示意图片 1.png，继续执行后续实验。')
    print('教学提示：即使没有图片，本实验仍可以完整完成数据读取、建模和分析。')

### 第二步：认识原始数据

真实交易数据通常很大、很复杂，直接全部拿来做课堂实验，初学者很容易被数据规模淹没。

所以这里会先抽取和整理一个更适合教学演示的数据子集，重点是让大家看懂流程：

1. 交易记录长什么样？
2. 标签列表示什么？
3. 欺诈样本和正常样本数量差多少？

可以把这一步理解成：先用“教学版样本”练会方法，再考虑更大规模的数据。

In [ ]:
# 读取 csv 文件，并为课堂实验构造一个较易运行的平衡子集
selected_columns = [
    'step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'nameOrig', 'nameDest',
]

fraud_chunks = []
normal_chunks = []
for chunk in pd.read_csv('financial_data.csv', usecols=selected_columns, chunksize=100000):
    fraud_chunk = chunk[chunk['isFraud'] == 1]
    normal_chunk = chunk[chunk['isFraud'] == 0]
    if not fraud_chunk.empty:
        fraud_chunks.append(fraud_chunk)
    if not normal_chunk.empty:
        normal_chunks.append(normal_chunk.sample(n=min(400, len(normal_chunk)), random_state=42))

fraud_data = pd.concat(fraud_chunks, ignore_index=True)
normal_data = pd.concat(normal_chunks, ignore_index=True)
data = pd.concat([fraud_data, normal_data], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

data['isFraud'] = data['isFraud'].astype(int)
data['isFlaggedFraud'] = data['isFlaggedFraud'].astype(int)

print('教学用子集样本数, 字段数:', data.shape)
print('\n类别分布:')
print(data['isFraud'].value_counts())
print('\n前 5 行数据:')
print(data.head())
print('\n教学提示：原始文件很大，课堂实验先抽取一个更容易运行的本地子集，后续有兴趣可以再尝试扩大规模。')

### 第三步：理解原始特征并构造关键风险信号

和医学分类教程先处理异常体征一样，这个金融分类教程也需要先把“哪些字段真正携带风险信息”想清楚，再交给模型学习。

1. 单看交易金额还不够，还要结合交易前后余额变化判断这笔交易是否合理。
2. `type` 这类字段是文本，不能直接输入模型，但它本身又是重要的风险线索。
3. 通过构造差额、比例、高风险类型标记等衍生特征，可以把“交易行为是否异常”表达得更清楚。
4. 这一步的目标不是把字段越堆越多，而是把原始交易记录整理成更有判别力的风险信号。

### 原始特征与补充特征含义

金融交易表里的字段看起来很多，但如果不知道每一列代表什么，就很难理解为什么有些交易更可疑、为什么要补充“差额”“比例”之类的新特征。

**原始特征含义：**

1. `step`：交易发生的时间步，可以粗略理解为交易所处时段。
2. `type`：交易类型，如转账、提现、支付等。
3. `amount`：交易金额。
4. `oldbalanceOrg`：转出账户交易前余额。
5. `newbalanceOrig`：转出账户交易后余额。
6. `oldbalanceDest`：转入账户交易前余额。
7. `newbalanceDest`：转入账户交易后余额。
8. `isFlaggedFraud`：系统是否对该交易做了高风险标记。
9. `isFraud`：该交易是否为欺诈，是本任务要判断的标签。

**补充特征含义：**

1. `origin_balance_change`：转出账户余额变化量，帮助判断扣款是否合理。
2. `dest_balance_change`：转入账户余额变化量，帮助观察收款侧是否异常。
3. `amount_to_origin_balance`：交易金额与转出账户原余额的比例，用来判断这笔交易是否“过大”。
4. `amount_to_dest_balance`：交易金额与目标账户原余额的比例，用来观察资金流入是否突兀。
5. `is_transfer_or_cashout`：是否属于高风险的转账/提现类型。
6. `origin_zero_before`：转出账户在交易前是否几乎没有余额。
7. `dest_zero_before`：转入账户在交易前是否几乎没有余额。
8. `origin_balance_error`：按金额推算后的转出账户余额误差，用来发现不一致记录。
9. `dest_balance_error`：按金额推算后的转入账户余额误差。
10. `type_*`：对交易类型做独热编码后得到的数值特征。

这些补充特征的核心思想是：不仅看“发生了一笔多少钱的交易”，还要看“这笔钱相对于账户原状态是否异常、交易前后变化是否说得通”。

### 第四步：完成编码、标准化、类别平衡与张量化

这一格和医学分类教程保持同样的操作目的，都是把原始记录一步步整理成神经网络能够学习的训练输入。

1. **编码（Encoding）**：把 `type` 这样的类别字段转换成数值形式，便于模型比较不同交易类型的风险模式。
2. **标准化（Standardization）**：金额、余额、比例等字段范围差异很大，标准化后训练更稳定。
3. **类别平衡（Class Balancing）**：欺诈样本通常远少于正常交易，如果不做平衡处理，模型容易偏向多数类。
4. **张量化（Tensorization）**：把整理后的数组转换成 PyTorch 张量，后续才能执行前向预测、误差计算和反向更新。

可以把这一步理解成：先把交易记录翻译成统一尺度的风险特征，再交给模型正式学习。

In [ ]:
# 构造衍生特征、完成编码与标准化，并根据情况做类别平衡
data_model = data.copy()

data_model['origin_balance_change'] = data_model['oldbalanceOrg'] - data_model['newbalanceOrig']
data_model['dest_balance_change'] = data_model['newbalanceDest'] - data_model['oldbalanceDest']
data_model['amount_to_origin_balance'] = data_model['amount'] / (data_model['oldbalanceOrg'] + 1)
data_model['amount_to_dest_balance'] = data_model['amount'] / (data_model['oldbalanceDest'] + 1)
data_model['is_transfer_or_cashout'] = data_model['type'].isin(['TRANSFER', 'CASH_OUT']).astype(int)
data_model['origin_zero_before'] = (data_model['oldbalanceOrg'] == 0).astype(int)
data_model['dest_zero_before'] = (data_model['oldbalanceDest'] == 0).astype(int)
data_model['origin_balance_error'] = np.abs((data_model['oldbalanceOrg'] - data_model['amount']) - data_model['newbalanceOrig'])
data_model['dest_balance_error'] = np.abs((data_model['oldbalanceDest'] + data_model['amount']) - data_model['newbalanceDest'])

data_model = pd.get_dummies(data_model, columns=['type'], drop_first=True)

X = data_model.drop(['isFraud', 'nameOrig', 'nameDest'], axis=1).astype(float)
y = data_model['isFraud'].values

print('建模字段数:', X.shape[1])
print('\n前 5 行特征:')
print(X.head())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if HAS_SMOTE:
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_scaled, y)
    print('\n已使用 SMOTE 进一步平衡类别。')
else:
    X_resampled, y_resampled = X_scaled, y
    print('\n当前环境未安装 imblearn，跳过 SMOTE，直接使用采样后的数据继续实验。')

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

print('\n训练集形状:', X_train_tensor.shape, y_train_tensor.shape)
print('测试集形状:', X_test_tensor.shape, y_test_tensor.shape)
print('\n教学提示：这里不只保留原始交易字段，还补充了交易前后差额、比例和高风险类型标记，让模型更容易学习交易行为模式。')

#### 特征降维可视化

 > 在正式建模前，先把高维交易特征压缩到三维空间，观察正常交易和欺诈交易是否出现明显的分布差异。

 > 现在改为三维交互式散点图，你可以在 notebook 输出中直接旋转、缩放和悬停查看局部样本。

 > 如果两类样本依然大量重叠，就说明仅靠当前字段还难以完全分开它们，后续可以继续思考更有效的特征工程。

In [ ]:
# 用 PCA 把交易特征压缩到三维空间，观察正常交易与欺诈交易的分布
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_resampled)

pca_df = pd.DataFrame({
    '主成分1': X_pca[:, 0],
    '主成分2': X_pca[:, 1],
    '主成分3': X_pca[:, 2],
    '类别': np.where(y_resampled == 1, '欺诈交易', '正常交易'),
    '交易金额': X_resampled[:, 2],
    '原账户余额': X_resampled[:, 3],
    '目标账户旧余额': X_resampled[:, 5],
})

fig = px.scatter_3d(
    pca_df,
    x='主成分1',
    y='主成分2',
    z='主成分3',
    color='类别',
    hover_data=['交易金额', '原账户余额', '目标账户旧余额'],
    title='金融分类任务：特征降维后的三维交互分布',
    opacity=0.58,
    color_discrete_map={'正常交易': '#4c956c', '欺诈交易': '#d1495b'},
 )
fig.update_traces(marker=dict(size=3))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig.show()

explained_ratio = pca.explained_variance_ratio_
print('前三个主成分的方差解释率:', np.round(explained_ratio, 4))
print('累计解释率:', round(float(explained_ratio.sum()), 4))
print('教学提示：你可以旋转图形，观察欺诈样本是否在三维空间里形成相对聚集区域。')

In [ ]:
# 学生可在这里修改训练参数，再重新运行后续单元
student_config = {
    'hidden_dims': [32, 16],
    'learning_rate': 0.001,
    'epochs': 200,
    'threshold': 0.5,
}

print('当前实验参数:', student_config)

### 第五步：开始训练 MLP 分类模型

完成数据准备后，模型才真正开始“学会区分正常交易和欺诈交易”。

这里的网络会输出一个 0 到 1 之间的概率，可以把它理解成“这笔交易看起来像欺诈的程度”。

训练时，模型会不断根据预测错误来调整参数，让欺诈样本的概率更高、正常样本的概率更低。

In [ ]:
# 定义模型，损失函数，优化器
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        layers.append(nn.Sigmoid())
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

hidden_dims = student_config.get('hidden_dims', [32, 16])
learning_rate = student_config.get('learning_rate', 0.001)
num_epochs = student_config.get('epochs', 200)
threshold = student_config.get('threshold', 0.5)

model = MLP(X_train.shape[1], hidden_dims)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# 训练模型
for epoch in trange(num_epochs, desc="Training Epochs"):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor).squeeze()
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()

In [ ]:
# 评估模型
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor).squeeze()
    predicted_classes = (test_outputs > threshold).float()
    confusion = confusion_matrix(y_test_tensor.numpy(), predicted_classes.numpy())
    accuracy = accuracy_score(y_test_tensor.numpy(), predicted_classes.numpy())
    print(f'Accuracy: {accuracy:.4f}')
    print(f'当前参数: hidden_dims={hidden_dims}, learning_rate={learning_rate}, epochs={num_epochs}, threshold={threshold}')

    print('\nConfusion Matrix:')
    print(confusion)
    print('\nClassification Report:')
    print(classification_report(y_test_tensor.numpy(), predicted_classes.numpy(), target_names=['正常交易', '欺诈交易']))

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(confusion, cmap='Greens')
    ax.set_title('混淆矩阵热力图')
    ax.set_xlabel('预测类别')
    ax.set_ylabel('真实类别')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['正常交易', '欺诈交易'])
    ax.set_yticklabels(['正常交易', '欺诈交易'])

    for row_index in range(confusion.shape[0]):
        for column_index in range(confusion.shape[1]):
            ax.text(column_index, row_index, int(confusion[row_index, column_index]), ha='center', va='center', color='black')

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

### 第六步：查看分类结果

在类别不平衡任务里，只看一个总体准确率尤其危险。

因为模型就算把大多数样本都判成正常交易，也可能看起来“挺准”，但真正重要的欺诈交易却漏掉了。

所以这里更应该关注：

1. 欺诈类的识别率如何？
2. 正常类和欺诈类谁更容易被误判？
3. 混淆矩阵里错误主要集中在哪一格？

In [ ]:
# 可视化预测结果
%matplotlib inline
probabilities = test_outputs.numpy()
predictions_np = predicted_classes.numpy()
y_test_np = y_test_tensor.numpy()

plt.figure(figsize=(10, 4))
sample_count = min(30, len(y_test_np))
indices = np.arange(sample_count)
plt.plot(indices, y_test_np[:sample_count], 'o-', label='真实标签')
plt.plot(indices, probabilities[:sample_count], 's--', label='预测为欺诈的概率')
plt.title('金融欺诈检测：前 30 个测试样本预测结果')
plt.xlabel('测试样本序号')
plt.ylabel('标签 / 概率')
plt.legend()
plt.tight_layout()
plt.show()

wrong_indices = np.where(predictions_np != y_test_np)[0]
print('误判样本数:', len(wrong_indices))
if len(wrong_indices) > 0:
    first_wrong = int(wrong_indices[0])
    print('\n示例误判样本：')
    print('真实标签:', '欺诈交易' if y_test_np[first_wrong] == 1 else '正常交易')
    print('预测标签:', '欺诈交易' if predictions_np[first_wrong] == 1 else '正常交易')
    print('预测为欺诈的概率:', float(probabilities[first_wrong]))

### 第七步：分析概率区间误判

前面已经知道模型总体表现，这一步继续追问“错在什么地方”。

重点观察：

1. 接近阈值的样本是不是最容易摇摆？
2. 哪些概率区间里的预测最不稳定？
3. 如果调整阈值，误判和漏判会怎么变化？

这一步能帮助学生理解：分类模型给出的不是“绝对结论”，而是带有置信程度的判断。

In [ ]:
# 概率区间误差分析：观察不同预测概率区间的误判分布
probability_error = np.abs(probabilities - y_test_np)
analysis_df = pd.DataFrame({
    '真实标签': y_test_np.astype(int),
    '预测标签': predictions_np.astype(int),
    '预测概率': probabilities,
    '概率误差': probability_error,
})
analysis_df['概率区间'] = pd.cut(
    analysis_df['预测概率'],
    bins=np.linspace(0, 1, 6),
    include_lowest=True,
 )

summary_rows = []
for interval_label, group in analysis_df.groupby('概率区间', observed=False):
    if len(group) == 0:
        continue
    summary_rows.append({
        '概率区间': interval_label,
        '样本数': len(group),
        '平均概率误差': round(group['概率误差'].mean(), 3),
        '中位概率误差': round(group['概率误差'].median(), 3),
        '误判率': round((group['预测标签'] != group['真实标签']).mean(), 3),
        '最大概率误差': round(group['概率误差'].max(), 3),
    })

error_summary = pd.DataFrame(summary_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(probabilities[y_test_np == 0], bins=np.linspace(0, 1, 11), alpha=0.65, label='真实正常交易', color='#4c956c')
axes[0].hist(probabilities[y_test_np == 1], bins=np.linspace(0, 1, 11), alpha=0.65, label='真实欺诈交易', color='#d1495b')
axes[0].set_xlabel('预测为欺诈的概率')
axes[0].set_ylabel('样本数')
axes[0].set_title('不同真实类别的预测概率分布')
axes[0].legend()

boxplot_data = [
    analysis_df.loc[analysis_df['概率区间'] == interval_label, '概率误差'].values
    for interval_label in error_summary['概率区间']
]
axes[1].boxplot(boxplot_data, patch_artist=True)
axes[1].set_xticks(range(1, len(error_summary['概率区间']) + 1))
axes[1].set_xticklabels([str(label) for label in error_summary['概率区间']])
axes[1].set_xlabel('预测概率区间')
axes[1].set_ylabel('概率误差 |真实标签 - 预测概率|')
axes[1].set_title('不同概率区间的预测误差分布')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print('不同预测概率区间的误差统计：')
print(error_summary)
print('\n教学提示：如果某些概率区间误判率很高，说明模型在这些区间里的判断还不够稳定。')

#### 特征重要性反思与验证

> 这里同样使用置乱测试：每次只打乱一个特征，再观察准确率下降了多少。

> 下降越明显，说明当前模型越依赖这个特征。

In [ ]:
# 用置乱测试验证分类任务中的特征重要性
baseline_accuracy = float(accuracy)
X_test_for_importance = X_test.copy()
importance_rows = []

for feature_index, feature_name in enumerate(X.columns):
    shuffled_X_test = X_test_for_importance.copy()
    np.random.seed(42)
    np.random.shuffle(shuffled_X_test[:, feature_index])
    shuffled_tensor = torch.FloatTensor(shuffled_X_test)

    with torch.no_grad():
        shuffled_probabilities = model(shuffled_tensor).squeeze().numpy()
        shuffled_predictions = (shuffled_probabilities > threshold).astype(float)

    shuffled_accuracy = accuracy_score(y_test, shuffled_predictions)
    importance_rows.append({
        '特征': feature_name,
        '打乱后准确率': round(float(shuffled_accuracy), 4),
        '准确率下降值': round(float(baseline_accuracy - shuffled_accuracy), 4),
    })

importance_df = pd.DataFrame(importance_rows).sort_values('准确率下降值', ascending=False)
print('基线 Accuracy:', round(baseline_accuracy, 4))
print('\n特征重要性验证结果：')
print(importance_df)

plt.figure(figsize=(8, 5))
plt.barh(importance_df['特征'], importance_df['准确率下降值'], color='#4c956c')
plt.gca().invert_yaxis()
plt.xlabel('打乱该特征后下降的 Accuracy')
plt.ylabel('特征')
plt.title('金融分类任务：置乱测试下的特征重要性')
plt.tight_layout()
plt.show()

print('教学提示：准确率下降值越大，说明模型越依赖该特征。')
print('注意：这只是教学版的重要性验证，不等于严格的因果解释。')

In [ ]:
# 选做：观察不同阈值对分类结果的影响
threshold_candidates = [0.3, 0.4, 0.5, 0.6, 0.7]
threshold_results = []

for threshold_candidate in threshold_candidates:
    predicted_candidate = (test_outputs > threshold_candidate).float()
    candidate_accuracy = accuracy_score(y_test_tensor.numpy(), predicted_candidate.numpy())
    threshold_results.append({
        'threshold': threshold_candidate,
        'accuracy': round(candidate_accuracy, 4),
        'positive_predictions': int(predicted_candidate.sum().item()),
    })

threshold_results_df = pd.DataFrame(threshold_results)
print(threshold_results_df)
print('\n思考：阈值越低，模型越容易把样本判成欺诈；阈值越高，则越保守。')

#### 结果反思与动手挑战

请同学们结合本次运行结果尝试回答下面几个问题：

1. 本次运行 Accuracy 约为 0.94，但仍有 623 个误判样本。为什么在风控任务里，高 Accuracy 仍然不能说明问题已经解决？
2. 从概率区间统计看，`0.4-0.6` 区间的误判率最高。这和“接近阈值的样本更难判断”这一结论是否一致？
3. 置乱测试里，`type_TRANSFER`、`origin_balance_error`、`origin_zero_before` 排在前面，这说明我们新增的交易行为衍生特征起作用了吗？
4. 本次阈值实验里，默认的 `0.5` 准确率最高。为什么在这个案例里继续提高阈值并没有变得更好？
5. 如果降低 threshold，会更容易抓到欺诈交易还是更容易误报？你会如何在召回和误报之间做取舍？
6. 在三维交互图中，欺诈样本是否形成局部聚集？这和高重要性特征反映出的交易模式能否对应起来？
7. 如果继续扩充特征，你最想增加哪类字段：账户关系、时间模式，还是地理/设备信息？为什么？

<details>
<summary>提示与参考答案</summary>

提示：从“三维投影分布”“类别不平衡”“阈值变化”“误判和漏判代价”“特征重要性验证”几个角度联合分析。

参考答案：
1. Accuracy 约 0.94 说明模型整体已经较强，但 623 个误判样本在真实风控场景里仍可能带来明显业务成本，所以不能只看总分数。
2. `0.4-0.6` 区间最接近分类边界，模型在这里最不确定，因此误判率最高是合理现象。
3. `type_TRANSFER`、`origin_balance_error`、`origin_zero_before` 排在前面，说明新增的交易行为衍生特征确实帮助模型捕捉到了更有区分力的模式。
4. 这次运行里 0.5 的 Accuracy 最高，说明继续调高阈值会让模型变得过于保守，错过部分本应判成欺诈的样本。
5. 降低阈值会让模型更容易判成欺诈，通常有利于提高召回，但也会增加误报，需要结合人工审核成本一起权衡。
6. 如果三维图中欺诈样本形成局部聚集，就说明当前特征已经提取出一部分可分模式；如果仍有重叠，也说明还有继续补充特征的空间。
7. 账户关系、时间模式和设备信息都可能有价值，其中最值得优先补充的是能直接揭示异常交易链路或异常行为节奏的字段。
</details>